In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2 as pg2
from datetime import datetime

In [ ]:
host = 'localhost'
port = '5432'
database = 'DWH'
user = 'test'
password = '123mat'

In [13]:
cursor = conn.cursor() 

# Ejecutamos nuestra sentencia SQL
# cursor.execute("INSERT INTO snowflake.dim_terminales (terminal, tipo, nombre, departamento, ubicacion, direccion) VALUES('AT28', 'NORMAL', 'FARMACIA SONSONATE IV', 'SV-SO', 'FARMACIA', 'SONSONATE');")
cursor.execute("DELETE FROM snowflake.dim_terminales WHERE TERMINAL = 'AT27';")

# Confirmamos los cambios en la base de datos
conn.commit()

# Cerramos el cursor y la conexión
cursor.close()
conn.close()


In [22]:
q = 'SELECT * FROM snowflake.fact_transaccionesatm A left join snowflake.dim_terminales B on A.terminal = B.terminal;'
df = pd.read_sql(q, conn)
df

df01 = df[["nombre","monto"]].groupby(["nombre"]).sum()
df01

C:\Users\celta\AppData\Local\Temp\ipykernel_20124\2590188738.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(q, conn)


,monto
nombre,
CENTRO COMERCIAL COJUTEPEQUE,56095
CENTRO COMERCIAL SAN MIGUEL,58380
CENTRO COMERCIAL SANTA ANA,49935
CENTRO COMERCIAL SENSUNTEPEQUE,46840
CENTRO COMERCIAL SONSONATE,56795
FARMACIA AHUACHAPÁN,57860
FARMACIA CHALATENANGO,52565
FARMACIA SAN VICENTE,53595
FARMACIA SANTA TECLA,53260


In [26]:
conn = pg2.connect(host=host, port=port, database=database, user=user, password=password)
cursor = conn.cursor()
cursor.execute("insert into snowflake.test(nombre,monto) values ('PRUEBA 01', 100);")
conn.commit()
cursor.close() 
conn.close()

In [ ]:
conn = pg2.connect(host=host, port=port, database=database, user=user, password=password)
cursor = conn.cursor()
cursor.execute("insert into snowflake.test(nombre,monto) values (%s, %s);", ('PRUEBA 02', 200)) # Usando placeholders para mayor seguridad
conn.commit()
cursor.close() 
conn.close()

In [41]:
# crear una lista de tuplas con la informacion del df
datos = list(df01.itertuples(index=True, name=None))

conn = pg2.connect(host=host, port=port, database=database, user=user, password=password)
cursor = conn.cursor()
cursor.executemany("insert into snowflake.test(nombre, monto) values (%s, %s);", datos)

conn.commit()


In [52]:
df02 = df01
df02["datetime"] = datetime.now() # Agregar una columna con la fecha y hora actual
df02

# datos = list(df01.itertuples(index=True, name=None))
# datos

,monto,datetime
nombre,,
CENTRO COMERCIAL COJUTEPEQUE,56095,2025-11-24 20:22:23.813828
CENTRO COMERCIAL SAN MIGUEL,58380,2025-11-24 20:22:23.813828
CENTRO COMERCIAL SANTA ANA,49935,2025-11-24 20:22:23.813828
CENTRO COMERCIAL SENSUNTEPEQUE,46840,2025-11-24 20:22:23.813828
CENTRO COMERCIAL SONSONATE,56795,2025-11-24 20:22:23.813828
FARMACIA AHUACHAPÁN,57860,2025-11-24 20:22:23.813828
FARMACIA CHALATENANGO,52565,2025-11-24 20:22:23.813828
FARMACIA SAN VICENTE,53595,2025-11-24 20:22:23.813828
FARMACIA SANTA TECLA,53260,2025-11-24 20:22:23.813828


In [53]:
# crear una lista de tuplas con la informacion del df
datos = list(df01.itertuples(index=True, name=None))

conn = pg2.connect(host=host, port=port, database=database, user=user, password=password)
cursor = conn.cursor()
cursor.executemany("insert into snowflake.test(nombre, monto, timestamp) values (%s, %s, %s);", datos)
conn.commit()
cursor.close()
conn.close()